# Phase 6 (Summarisation) — c-6 simple encoder-decoder + LORMA

LORMA here is your multiplicative layer **W' = (I + BA) W** (not LoRA's W + BA).

Run the cells in order. **Only Cell E2 section [H] needs filling** (every value marked `# <<< FILL`). Kaggle: GPU on, internet on (HF dataset, `rouge-score`, `bert-score`), HF secret `HF`, attach your Phase 1 c-6 notebook output and set `ROOT` in Cell A.

## Cell 0 — Phase 1 imports, HF login, tokenizer class, model classes

In [1]:
# --- Phase 1 c-6 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict


# HF login (Phase 1 cell 2), needed if the dataset download asks for auth
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
# --- Phase 1 c-6 notebook, cell 8 (verbatim): SPECIAL_TOKENS, BPETokenizer, train_tokenizer ---


SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [3]:
# --- Phase 1 c-6 notebook, cells 20, 22 and 23 (verbatim, in this order) ---
# Cell 23 must come after cell 22: it redefines CausalSelfAttention (q/k/v_proj), which the
# saved decoder weights use. Without it, Cell A fails with qkv_proj key errors in decoder_layers.

# ---- cell 20 ----
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---- cell 22 ----
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Seq2SeqConfig:
    vocab_size: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    n_layers: int = 8

# ---------------------------------------------------------
# 1. Attention Modules
# ---------------------------------------------------------
class EncoderSelfAttention(nn.Module):
    """Bidirectional self-attention for the encoder (no causal mask)."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CausalSelfAttention(nn.Module):
    """Causal self-attention for the decoder (looks only at past tokens)."""
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv_proj(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in qkv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.mask[:T, :T], float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class CrossAttention(nn.Module):
    """Cross-attention: Query from Decoder, Key/Value from Encoder."""
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        B_c, T_c, C_c = context.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(context).chunk(2, dim=-1)
        k, v = [t.view(B_c, T_c, self.n_heads, self.d_head).transpose(1, 2) for t in kv]

        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

# ---------------------------------------------------------
# 2. Feed-Forward & Standard Blocks
# ---------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class EncoderBlock(nn.Module):
    """Standard bidirectional Transformer encoder block."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = EncoderSelfAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Standard causal Transformer decoder block with cross-attention."""
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout)
        
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        x = x + self.self_attn(self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), encoder_out)
        x = x + self.ffn(self.ln3(x))
        return x


# ---------------------------------------------------------
# 4. Hourglass Blocks (Encoder & Decoder versions)
# ---------------------------------------------------------
class EncoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([EncoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor):
        residual = x
        shortened_x = self.downsample(x) # No shift for Encoder
        for block in self.bottleneck:
            shortened_x = block(shortened_x)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual


class DecoderHourglassBlock(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig, factor: int, bottleneck_layers: int):
        super().__init__()
        self.factor = factor
        self.downsample = Downsample(factor)
        self.bottleneck = nn.ModuleList([DecoderBlock(cfg) for _ in range(bottleneck_layers)])
        self.upsample = Upsample(factor)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor):
        residual = x
        x_shifted = shift_right(x, shift=self.factor - 1) # Shift required for Decoder causality
        shortened_x = self.downsample(x_shifted)
        for block in self.bottleneck:
            shortened_x = block(shortened_x, encoder_out)
        expanded_x = self.upsample(shortened_x)
        return expanded_x + residual

# ---------------------------------------------------------
# 5. Top-Level Hourglass Seq2Seq Model
# ---------------------------------------------------------
class StandardEncoderDecoder(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)

        self.encoder_layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.encoder_ln = nn.LayerNorm(cfg.d_model)

        self.decoder_layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.decoder_ln = nn.LayerNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def get_positional_embeddings(self, seq_len, device):
        return self.pos_emb(torch.arange(seq_len, device=device).unsqueeze(0))

    def encode(self, enc_idx):
        x = self.tok_emb(enc_idx) + self.get_positional_embeddings(enc_idx.size(1), enc_idx.device)
        for block in self.encoder_layers:
            x = block(x)
        return self.encoder_ln(x)

    def decode(self, dec_idx, encoder_out):
        x = self.tok_emb(dec_idx) + self.get_positional_embeddings(dec_idx.size(1), dec_idx.device)
        for block in self.decoder_layers:
            x = block(x, encoder_out)
        return self.lm_head(self.decoder_ln(x))

    def forward(self, enc_idx, dec_idx):
        return self.decode(dec_idx, self.encode(enc_idx))


# ---- cell 23 ----
# ============================================================
# CELL 1 — Decoder-only Transformer (your A3 Q1 code, unchanged except the 2 marked LayerNorm lines)
# Replaces: §9 "Define The Model" (GPT2Config / GPT2LMHeadModel) and the DataCollator cell.
# ============================================================
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------- A3 Q1.1: manual causal multi-head self-attention ----------
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


# ---------- A3 Q1.2: decoder block and language model ----------
@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 2000
    max_seq_len: int = 256
    dropout: float = 0.15
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


@dataclass
class EncDecConfig(Seq2SeqConfig):
    enc_len: int = None      # tokens the encoder reads (prompt side)
    dec_len: int = None      # tokens the decoder learns to produce (continuation side)


class EncDecLM(nn.Module):
    def __init__(self, cfg: EncDecConfig):
        super().__init__()
        assert None not in (cfg.enc_len, cfg.dec_len), "set enc_len and dec_len in the config"
        self.cfg = cfg
        self.core = StandardEncoderDecoder(cfg)

    def encode(self, enc_idx):
        return self.core.encode(enc_idx)

    def decode(self, dec_idx, encoder_out):
        return self.core.decode(dec_idx, encoder_out)

    def forward(self, enc_idx, dec_idx):
        return self.core(enc_idx, dec_idx)

## Cell A — load the Phase 1 c-6 model and tokenizer

In [4]:
# =====================================================================================
# CELL A — load the Phase 1 c-6 encoder-decoder and its tokenizer
# Expect <All keys matched successfully>, 55.13M params, vocab 11022, d_model 560, 8 layers, max_seq_len 1536
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudraalt/phase-1-c-6-simple-encoder-decoder"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/model_encdec/bpe_tokenizer.pkl", "rb"))   # c-6's own tokenizer
ck = torch.load(f"{ROOT}/cs781_phase1_full/model_encdec/model_final.pt", map_location="cpu", weights_only=False)
cfg = EncDecConfig(**ck["config"])
lm = EncDecLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| n_layers", cfg.n_layers, "| max_seq_len", cfg.max_seq_len)

<All keys matched successfully>
params 55.13008 M | vocab 11022 | d_model 560 | n_layers 8 | max_seq_len 1536


## Cell B — encoder sanity check

In [5]:
# =====================================================================================
# CELL B — per-token vectors from the c-6 ENCODER + sanity check
# The encoder is bidirectional: every token sees the tokens on both sides. The decoder is not used.
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 1536
    c = lm.core
    T = idx.size(1)
    assert T <= c.cfg.max_seq_len         # learned position table has 1536 rows
    x = c.tok_emb(idx) + c.get_positional_embeddings(T, idx.device)
    for blk in c.encoder_layers:          # 8 bidirectional EncoderBlocks
        x = blk(x)
    return c.encoder_ln(x)                # (B, T, 560)

# Sanity check: same as the model's own encoder
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f, ref = features(idx), lm.encode(idx)
print("max diff", (f - ref).abs().max().item(), tuple(f.shape))   # expect 0.0, (1, 64, 560)

max diff 0.0 (1, 64, 560)


## Cell C — Phase 6 data (train / held-out val / test)

In [6]:
# =====================================================================================
# CELL C — [ANCILLARY] Phase 6 (Summarisation) data: train / held-out val / test
# One record = one long judgment ("document") -> one reference summary ("summary").
# There is NO dev split in Phase 6, so a fixed, seeded part of train is held out as val.
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, random
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "Exploration-Lab/CS-781-Capstone"
P6_DIR = "Phase_6_Summ"
print("Phase 6 files on HF:", [f for f in list_repo_files(REPO, repo_type="dataset") if "Phase_6" in f])

VAL_FRAC   = 0.10   # share of train held out for validation (no official dev split)
SPLIT_SEED = 42     # fixed, so every run / backbone uses the SAME held-out documents

def get6(name):
    """Reads Phase_6_Summ/<name>. Accepts plain JSON or JSON-lines."""
    p = hf_hub_download(REPO, f"{P6_DIR}/{name}", repo_type="dataset")
    with open(p, encoding="utf-8") as f:
        txt = f.read()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

TEXT_FIELDS    = ("document", "text", "judgment", "case_text", "source")
SUMMARY_FIELDS = ("summary", "abstract", "target", "headnote")

def as_docs(raw, name):
    """-> {doc_id: {"text": str, "summary": str or None}}, in file order.
    Accepts a list of dicts, {doc_id: {...}}, a one-key wrapper like {"data": [...]},
    or columns {"id": [...], "document": [...], ("summary": [...])}.
    Repeated ids: exact duplicates dropped; differing repeats kept as "<id>#dup<n>" (train),
    dropped with a warning on test (exactly one prediction per id)."""
    while isinstance(raw, dict) and len(raw) == 1 and isinstance(next(iter(raw.values())), (list, dict)):
        raw = next(iter(raw.values()))
    if isinstance(raw, dict) and "id" in raw and isinstance(raw["id"], list):
        keys = list(raw)
        raw = [{k: raw[k][i] for k in keys} for i in range(len(raw["id"]))]
    items = [(d.get("id", n), d) for n, d in enumerate(raw)] if isinstance(raw, list) else list(raw.items())
    docs, n_exact, n_diff = {}, 0, 0
    for k, d in items:
        if isinstance(d, str):                       # {id: "document text"} layout
            d = {"document": d}
        assert isinstance(d, dict), f"doc {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        tf = next((f for f in TEXT_FIELDS if f in d), None)
        assert tf is not None, f"doc {k}: no text field (keys {list(d)})"
        text = d[tf]
        if isinstance(text, list):
            text = " ".join(map(str, text))
        sf = next((f for f in SUMMARY_FIELDS if f in d), None)
        summ = d[sf] if sf else None
        if isinstance(summ, list):
            summ = " ".join(map(str, summ))
        rec = {"text": str(text), "summary": None if summ is None else str(summ)}
        if k in docs:
            if docs[k] == rec:
                n_exact += 1
                continue
            if name == "test":
                print(f"WARNING test id {k} appears twice with different text: keeping the first")
                continue
            n_diff += 1
            j = 1
            while f"{k}#dup{j}" in docs:
                j += 1
            k = f"{k}#dup{j}"
        docs[k] = rec
    print(f"{name}: {len(items)} records -> {len(docs)} docs | exact duplicates dropped {n_exact} | "
          f"repeated ids with different content kept {n_diff}")
    return docs

_raw = get6("Train/train.json")
_first = _raw[0] if isinstance(_raw, list) else next(iter(_raw.values()))
print("train layout:", type(_raw).__name__, len(_raw), "| first record keys:",
      list(_first) if isinstance(_first, dict) else type(_first).__name__)
full_train = as_docs(_raw, "train")
test = as_docs(get6("Test/test.json"), "test")
del _raw

_empty = [k for k, d in full_train.items() if not (d["summary"] or "").strip()]
if _empty:
    print(f"dropping {len(_empty)} train docs with an empty summary")
    for k in _empty:
        del full_train[k]
_tl = sum(d["summary"] is not None for d in test.values())
if _tl:
    print(f"test has a summary field on {_tl} docs (ignored, set to None)")
for d in test.values():
    d["summary"] = None

# Held-out validation split (seeded shuffle of train ids)
_ids = list(full_train)
random.Random(SPLIT_SEED).shuffle(_ids)
_n_val = max(1, int(round(VAL_FRAC * len(_ids))))
VAL_IDS = set(_ids[:_n_val])
train = {k: d for k, d in full_train.items() if k not in VAL_IDS}
val   = {k: d for k, d in full_train.items() if k in VAL_IDS}
print(f"\ntrain {len(train)} | val (held out) {len(val)} | test {len(test)}")

for name, split in (("train", train), ("val", val), ("test", test)):
    w = np.array([len(d["text"].split()) for d in split.values()])
    line = f"{name:5s} document words: median {np.median(w):.0f} p90 {np.percentile(w, 90):.0f} max {w.max()}"
    if split is not test:
        s = np.array([len(d["summary"].split()) for d in split.values()])
        r = s / np.maximum(1, w)
        line += (f" | summary words: min {s.min()} median {np.median(s):.0f} p90 {np.percentile(s, 90):.0f} "
                 f"max {s.max()} | summary/document ratio median {np.median(r):.3f}")
    print(line)
print("first test ids:", list(test)[:3])
print(f"train∩test ids {len(set(full_train) & set(test))}")

Phase 6 files on HF: ['Phase_6_Summ/Test/test.json', 'Phase_6_Summ/Train/train.json']


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

train layout: list 7030 | first record keys: ['id', 'document', 'summary']
train: 7030 records -> 7030 docs | exact duplicates dropped 0 | repeated ids with different content kept 0


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

test: 100 records -> 100 docs | exact duplicates dropped 0 | repeated ids with different content kept 0
dropping 2 train docs with an empty summary

train 6325 | val (held out) 703 | test 100
train document words: median 3124 p90 7817 max 139645 | summary words: min 26 median 637 p90 1487 max 27768 | summary/document ratio median 0.217
val   document words: median 3205 p90 8316 max 90410 | summary words: min 69 median 659 p90 1651 max 8894 | summary/document ratio median 0.216
test  document words: median 3171 p90 8563 max 39695
first test ids: ['4963', '5994', '7130']
train∩test ids 0


## Cell D — tokenisation, ROUGE / BERTScore scorer, prediction.json writer, lead baseline

In [7]:
# =====================================================================================
# CELL D — [ANCILLARY] tokenisation, local ROUGE / BERTScore scorer, prediction.json writer, baseline
# Nothing in this cell is a model design or a hyperparameter choice.
# =====================================================================================
import json, os, sys, zipfile, subprocess, importlib
import numpy as np
from tqdm.auto import tqdm

PAD_ID = SPECIAL_TOKENS.index("<pad>")   # 0
UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
BOS_ID = SPECIAL_TOKENS.index("<s>")     # 2
EOS_ID = SPECIAL_TOKENS.index("</s>")    # 3
NUM_SPECIAL = len(SPECIAL_TOKENS)

def encode_text(text):
    ids = tok.encode(text)
    return ids if ids else [UNK_ID]

def ids_to_text(ids):
    """Generated ids -> text: stop at the first </s>, drop special tokens."""
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID)]
    return tok.decode([t for t in ids if t >= NUM_SPECIAL]).strip()

# Tokenise once: d["ids"] = whole judgment, d["sum_ids"] = whole reference summary (train/val)
for name, split in (("train", train), ("val", val), ("test", test)):
    for d in tqdm(split.values(), desc=f"tokenising {name}"):
        d["ids"] = encode_text(d["text"])
        if d["summary"] is not None:
            d["sum_ids"] = encode_text(d["summary"])

# Length statistics in c-6 tokens, to choose the encoder windows and DEC_LEN in Cell E2
for name, split in tqdm((("train", train), ("val", val), ("test", test)), desc="length stats", leave=False):
    L = np.array([len(d["ids"]) for d in split.values()])
    line = (f"{name:5s} doc tokens: median {np.median(L):.0f} p90 {np.percentile(L, 90):.0f} "
            f"p99 {np.percentile(L, 99):.0f} max {L.max()} | > 1536: {(L > 1536).mean():.1%}")
    if split is not test:
        S = np.array([len(d["sum_ids"]) for d in split.values()])
        line += (f" || summary tokens: median {np.median(S):.0f} p90 {np.percentile(S, 90):.0f} "
                 f"p99 {np.percentile(S, 99):.0f} max {S.max()} | > 1536: {(S > 1536).mean():.1%}")
    print(line)
_words = np.array([len(d["summary"].split()) for d in train.values()])
_toks = np.array([len(d["sum_ids"]) for d in train.values()])
print(f"c-6 tokens per summary word (train): {(_toks.sum() / max(1, _words.sum())):.2f}")


# -------------------------------------------------------------------------------------
# Local scorer: BERTScore F1 (primary), ROUGE-1 F1 and ROUGE-L F1 (secondary).
#   The official evaluator's BERTScore model/settings are not stated in the spec. Default here
#   is the bert_score package default for English (roberta-large, no baseline rescaling), the
#   usual choice; change BERTSCORE_MODEL if the TAs say otherwise. Scores are only comparable
#   to Codabench if the settings match, so use them for RELATIVE comparisons between runs.
#   These scorer models are used only to measure, never to train.
# -------------------------------------------------------------------------------------
def _ensure(pkg, pip_name):
    try:
        return importlib.import_module(pkg)
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)
        try:
            return importlib.import_module(pkg)
        except ImportError:
            print(f"  could not install {pip_name} (internet off?)")
            return None

_rouge_mod = _ensure("rouge_score", "rouge-score")
_bs_mod = _ensure("bert_score", "bert-score")
BERTSCORE_MODEL = None     # None = bert_score default for lang="en" (roberta-large)
BERTSCORE_BATCH = 16

if _rouge_mod is not None:
    from rouge_score import rouge_scorer
    _rs = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def _f(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0

def rouge_pair(pred, ref):
    """-> (ROUGE-1 F1, ROUGE-L F1). rouge_score package if available, else a plain re-implementation."""
    if _rouge_mod is not None:
        s = _rs.score(ref, pred)
        return s["rouge1"].fmeasure, s["rougeL"].fmeasure
    tp = re.findall(r"[a-z0-9]+", pred.lower())
    tr = re.findall(r"[a-z0-9]+", ref.lower())
    if not tp or not tr:
        return 0.0, 0.0
    cp, cr = Counter(tp), Counter(tr)
    ov = sum((cp & cr).values())
    l = _lcs(tp, tr)
    return _f(ov / len(tp), ov / len(tr)), _f(l / len(tp), l / len(tr))

_bs_scorer = None   # BERTScorer, loaded once and reused (bert_score.score() would reload roberta-large every call)

def bertscore_f1(preds, refs, chunk=256):
    """Per-pair BERTScore F1, computed in chunks with a tqdm bar."""
    global _bs_scorer
    if _bs_mod is None:
        return None
    from bert_score import BERTScorer
    if _bs_scorer is None:
        kw = dict(model_type=BERTSCORE_MODEL) if BERTSCORE_MODEL else dict(lang="en")
        _bs_scorer = BERTScorer(batch_size=BERTSCORE_BATCH,
                                device="cuda" if torch.cuda.is_available() else "cpu", **kw)
    preds = [p if p.strip() else "." for p in preds]
    out = []
    for i in tqdm(range(0, len(preds), chunk), desc="BERTScore", leave=False):
        _, _, F1 = _bs_scorer.score(preds[i:i + chunk], refs[i:i + chunk], verbose=False)
        out.append(F1.cpu().numpy())
    torch.cuda.empty_cache()
    return np.concatenate(out)

def evaluate_summaries(pred, docs, show=True, with_bertscore=True):
    """pred: {doc_id: summary str}. Scores every doc in docs that has a reference summary."""
    ids = [k for k in docs if docs[k].get("summary")]
    P = [pred[k] for k in ids]
    R = [docs[k]["summary"] for k in ids]
    r1, rl = zip(*[rouge_pair(p, r) for p, r in tqdm(zip(P, R), total=len(ids), desc="ROUGE", leave=False)])
    bs = bertscore_f1(P, R) if with_bertscore else None
    pw = np.array([len(p.split()) for p in P]); rw = np.array([len(r.split()) for r in R])
    res = {"bertscore_f1": None if bs is None else float(np.mean(bs)),
           "rouge1_f1": float(np.mean(r1)), "rougeL_f1": float(np.mean(rl)),
           "pred_words_median": float(np.median(pw)), "ref_words_median": float(np.median(rw)),
           "len_ratio_median": float(np.median(pw / np.maximum(1, rw))),
           "per_doc": {k: {"r1": a, "rl": b, "bs": None if bs is None else float(c), "pw": int(x), "rw": int(y)}
                       for k, a, b, c, x, y in zip(ids, r1, rl, bs if bs is not None else [None] * len(ids), pw, rw)}}
    if show:
        bstxt = "n/a" if res["bertscore_f1"] is None else f"{res['bertscore_f1']:.4f}"
        print(f"BERTScore F1 {bstxt} | ROUGE-1 F1 {res['rouge1_f1']:.4f} | ROUGE-L F1 {res['rougeL_f1']:.4f} | "
              f"words pred/ref median {res['pred_words_median']:.0f}/{res['ref_words_median']:.0f} "
              f"(ratio {res['len_ratio_median']:.2f}) | n={len(ids)}")
    return res

def repetition_rate(texts, n=3):
    """Share of word n-grams that repeat an earlier n-gram of the same summary (redundancy, for the report)."""
    rates = []
    for t in tqdm(list(texts), desc="repetition", leave=False):
        w = t.lower().split()
        g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
        if g:
            rates.append(1 - len(set(g)) / len(g))
    return float(np.mean(rates)) if rates else 0.0

def length_buckets(res, edges=(0.5, 0.8, 1.2, 1.5)):
    """Mean metrics by predicted/reference length ratio (how generation length moves the scores)."""
    rows = list(res["per_doc"].values())
    e = (0.0,) + tuple(edges) + (float("inf"),)
    print(f"{'pred/ref words':>16s} {'n':>5s} {'R-1':>6s} {'R-L':>6s} {'BS':>6s}")
    for lo, hi in zip(e[:-1], e[1:]):
        g = [r for r in rows if lo <= r["pw"] / max(1, r["rw"]) < hi]
        if g:
            bs = [r["bs"] for r in g if r["bs"] is not None]
            print(f"{lo:7.2f}-{hi:<8.2f} {len(g):5d} {np.mean([r['r1'] for r in g]):6.3f} "
                  f"{np.mean([r['rl'] for r in g]):6.3f} {np.mean(bs) if bs else float('nan'):6.3f}")


# -------------------------------------------------------------------------------------
# prediction.json writer: a JSON list of {"id": <test id>, "summary": str}, one per test id,
# ids exactly as in test.json, then submission.zip with prediction.json at the root.
# -------------------------------------------------------------------------------------
def write_prediction(pred, out_dir="/kaggle/working"):
    """pred: {test_id: summary str}."""
    os.makedirs(out_dir, exist_ok=True)
    missing = [k for k in test if k not in pred]
    extra = [k for k in pred if k not in test]
    assert not missing, f"{len(missing)} test ids have no summary, e.g. {missing[:3]}"
    assert not extra, f"{len(extra)} ids are not in test.json, e.g. {extra[:3]}"
    rows = []
    for k in test:
        s = pred[k]
        assert isinstance(s, str), f"{k}: summary must be a string"
        rows.append({"id": k, "summary": s if s.strip() else "."})   # never submit an empty string
    n_empty = sum(not pred[k].strip() for k in test)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    back = json.load(open(path, encoding="utf-8"))
    assert [r["id"] for r in back] == list(test) and len(back) == len(test)
    w = np.array([len(r["summary"].split()) for r in back])
    print(f"wrote {path} and {zpath}: {len(back)} summaries | words min {w.min()} median {np.median(w):.0f} "
          f"max {w.max()} | empty (written as '.') {n_empty}")
    return zpath


# -------------------------------------------------------------------------------------
# Baseline: lead-N words of the document, N = median train summary length in words.
# Any trained model should beat this on val.
# -------------------------------------------------------------------------------------
LEAD_N = int(np.median(_words))
lead_pred = {k: " ".join(d["text"].split()[:LEAD_N]) for k, d in tqdm(val.items(), desc="lead baseline", leave=False)}
print(f"\nlead-{LEAD_N} baseline on val:")
baseline_res = evaluate_summaries(lead_pred, val)


tokenising train:   0%|          | 0/6325 [00:00<?, ?it/s]

tokenising val:   0%|          | 0/703 [00:00<?, ?it/s]

tokenising test:   0%|          | 0/100 [00:00<?, ?it/s]

length stats:   0%|          | 0/3 [00:00<?, ?it/s]

train doc tokens: median 3903 p90 9778 p99 30822 max 172202 | > 1536: 91.7% || summary tokens: median 799 p90 1908 p99 5537 max 35431 | > 1536: 16.1%
val   doc tokens: median 3976 p90 10192 p99 29061 max 109893 | > 1536: 91.6% || summary tokens: median 825 p90 2069 p99 5318 max 11511 | > 1536: 18.2%
test  doc tokens: median 3886 p90 10873 p99 35854 max 48219 | > 1536: 95.0%
c-6 tokens per summary word (train): 1.28
installing bert-score ...


lead baseline:   0%|          | 0/703 [00:00<?, ?it/s]

... (9 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore:   0%|          | 0/3 [00:00<?, ?it/s]

BERTScore F1 0.8311 | ROUGE-1 F1 0.4939 | ROUGE-L F1 0.2473 | words pred/ref median 637/659 (ratio 0.97) | n=703


## Cell E0 — seq2seq plumbing (masked encoder/decoder, long-document windows, training, generation)

In [8]:
# =====================================================================================
# CELL E0 — [ANCILLARY] shared seq2seq plumbing for the c-6 Phase 6 (summarisation) experiments
# Run once after Cells 0, A, B, C, D. Then run Cell E2 (LORMA).
# Needs from earlier cells: lm, cfg, device, train, val, test (d["ids"], d["sum_ids"]),
#   PAD_ID, BOS_ID, EOS_ID, UNK_ID, ids_to_text, evaluate_summaries, write_prediction
#
# The c-6 encoder-decoder is used as a real seq2seq model:
#   judgment windows -> ENCODER -> memory -> DECODER (cross-attends to memory) -> summary tokens
#   training: teacher forcing, decoder input = <s> + summary[:-1], target = summary + </s>
#   inference: autoregressive decoding with a KV cache, controls set in Cell E2.
#
# Long documents (Phase 1 c-6 position table has 1536 rows, judgments are far longer):
#   windows of WINDOW tokens every STRIDE tokens (Phase 2/4/5 make_windows), DOC_REPR picks
#   which of them stand for the document ("all", "first", "last", "first_last", "spread"),
#   FUSION says how they reach the decoder:
#     "fid"    = each window is encoded SEPARATELY (positions 0..WINDOW-1 each), the encoder
#                outputs of all windows are concatenated and the decoder cross-attends to all
#                of them (Fusion-in-Decoder style; MAX_WINDOWS x WINDOW memory tokens).
#     "concat" = the selected windows' tokens are joined into ONE encoder input
#                (needs MAX_WINDOWS x WINDOW <= 1536); "first" + "concat" = plain truncation.
#
# The Phase 1 encoder and cross-attention have no padding mask. To batch documents of
# different lengths this cell runs the SAME blocks with the SAME weights (LORMA wrappers
# included, since every linear is called as a module) but adds a key-padding mask in
# encoder self-attention and in cross-attention. Decoder self-attention is causal, so padding
# at the end of a summary never reaches a real position. Self-checks at the end of the cell
# prove the masked / cached paths equal the original model.
# Nothing in this cell is a model design or a hyperparameter choice: every choice is an
# argument you set in Cell E2.
# =====================================================================================
import math, time, random, os, json
import numpy as np
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

FEAT_DIM = cfg.d_model          # 560 for c-6
MAX_POS = cfg.max_seq_len       # 1536: longest encoder window / decoder sequence
IGNORE = -100                   # target id that the loss skips (padding)
GRAD_CHECKPOINT = False         # set in Cell E2


# -------------------------------------------------------------------------------------
# Attention with masks (same maths as the Phase 1 modules)
# -------------------------------------------------------------------------------------
def _heads(t, B, T, H, D):
    return t.view(B, T, H, D).transpose(1, 2)            # (B, H, T, D)

def _drop_p(attn):
    return attn.attn_dropout.p if attn.training else 0.0

def _scale(attn):
    return getattr(attn, "attention_scale", None) or getattr(attn, "scale", None) or 1.0 / math.sqrt(attn.d_head)

def enc_self_attn(attn, x, key_mask):
    """EncoderSelfAttention.forward + key-padding mask. key_mask: (B, T) bool, True = real token, or None."""
    B, T, C = x.shape
    H, D = attn.n_heads, attn.d_head
    q, k, v = [_heads(t, B, T, H, D) for t in attn.qkv_proj(x).chunk(3, dim=-1)]
    m = None if key_mask is None else key_mask[:, None, None, :]
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, dropout_p=_drop_p(attn), scale=_scale(attn))
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

def _self_qkv(attn, x):
    if hasattr(attn, "qkv_proj"):                         # fused variant (Phase 1 cell 22)
        return attn.qkv_proj(x).chunk(3, dim=-1)
    return attn.q_proj(x), attn.k_proj(x), attn.v_proj(x)  # separate q/k/v (Phase 1 cell 23, the loaded one)

def dec_self_attn(attn, x, cache=None):
    """CausalSelfAttention.forward. With cache (dict) the new positions attend to cached past keys/values."""
    B, T, C = x.shape
    H, D = attn.n_heads, attn.d_head
    q, k, v = [_heads(t, B, T, H, D) for t in _self_qkv(attn, x)]
    if cache is not None and "k" in cache:
        k = torch.cat([cache["k"], k], dim=2)
        v = torch.cat([cache["v"], v], dim=2)
    if cache is not None:
        cache["k"], cache["v"] = k, v
    causal = T > 1                                         # one new token may see every cached position
    if causal and k.size(2) != T:
        raise ValueError("multi-token step with a cache is not supported")
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=_drop_p(attn), is_causal=causal, scale=_scale(attn))
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

def cross_kv(attn, memory):
    B, M, C = memory.shape
    k, v = attn.kv_proj(memory).chunk(2, dim=-1)
    return _heads(k, B, M, attn.n_heads, attn.d_head), _heads(v, B, M, attn.n_heads, attn.d_head)

def cross_attn(attn, x, kv, mem_mask):
    """CrossAttention.forward + memory padding mask. kv = cross_kv(attn, memory) (cached when decoding)."""
    B, T, C = x.shape
    q = _heads(attn.q_proj(x), B, T, attn.n_heads, attn.d_head)
    k, v = kv
    m = None if mem_mask is None else mem_mask[:, None, None, :]
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, dropout_p=_drop_p(attn), scale=_scale(attn))
    return attn.resid_dropout(attn.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))


# -------------------------------------------------------------------------------------
# Encoder / decoder passes (EncoderBlock / DecoderBlock .forward with the masked attention)
# -------------------------------------------------------------------------------------
def _enc_block(blk, x, key_mask):
    x = x + enc_self_attn(blk.self_attn, blk.ln1(x), key_mask)
    return x + blk.ffn(blk.ln2(x))

def _dec_block(blk, x, memory, mem_mask):
    x = x + dec_self_attn(blk.self_attn, blk.ln1(x))
    x = x + cross_attn(blk.cross_attn, blk.ln2(x), cross_kv(blk.cross_attn, memory), mem_mask)
    return x + blk.ffn(blk.ln3(x))

def _ckpt(backbone):
    return GRAD_CHECKPOINT and torch.is_grad_enabled() and backbone.training

def encoder_forward(backbone, idx, key_mask=None):
    """idx (N, T) padded with PAD_ID, key_mask (N, T) bool -> (N, T, 560)."""
    c = backbone.core
    assert idx.size(1) <= MAX_POS, f"encoder input of {idx.size(1)} tokens > {MAX_POS}"
    x = c.tok_emb(idx) + c.get_positional_embeddings(idx.size(1), idx.device)
    for blk in c.encoder_layers:
        x = checkpoint(_enc_block, blk, x, key_mask, use_reentrant=False) if _ckpt(backbone) else _enc_block(blk, x, key_mask)
    return c.encoder_ln(x)

def decoder_forward(backbone, dec_idx, memory, mem_mask=None):
    """Teacher-forced decoder: dec_idx (B, Td), memory (B, M, 560), mem_mask (B, M) -> logits (B, Td, V)."""
    c = backbone.core
    assert dec_idx.size(1) <= MAX_POS, f"decoder input of {dec_idx.size(1)} tokens > {MAX_POS}"
    x = c.tok_emb(dec_idx) + c.get_positional_embeddings(dec_idx.size(1), dec_idx.device)
    for blk in c.decoder_layers:
        x = checkpoint(_dec_block, blk, x, memory, mem_mask, use_reentrant=False) if _ckpt(backbone) else _dec_block(blk, x, memory, mem_mask)
    return c.lm_head(c.decoder_ln(x))


# -------------------------------------------------------------------------------------
# Which linears can be wrapped (encoder AND decoder), by short name
# -------------------------------------------------------------------------------------
def _dec_self_names(backbone):
    a = backbone.core.decoder_layers[0].self_attn
    return ("dec_self_qkv",) if hasattr(a, "qkv_proj") else ("dec_self_q", "dec_self_k", "dec_self_v")

def all_target_names(backbone):
    return (("enc_qkv", "enc_out", "enc_ffn_in", "enc_ffn_out") + _dec_self_names(backbone) +
            ("dec_self_out", "dec_cross_q", "dec_cross_kv", "dec_cross_out", "dec_ffn_in", "dec_ffn_out"))

_WHERE = {   # name -> (side, path inside the block)
    "enc_qkv": ("enc", ("self_attn", "qkv_proj")), "enc_out": ("enc", ("self_attn", "out_proj")),
    "enc_ffn_in": ("enc", ("ffn", "net", "0")), "enc_ffn_out": ("enc", ("ffn", "net", "2")),
    "dec_self_qkv": ("dec", ("self_attn", "qkv_proj")),
    "dec_self_q": ("dec", ("self_attn", "q_proj")), "dec_self_k": ("dec", ("self_attn", "k_proj")),
    "dec_self_v": ("dec", ("self_attn", "v_proj")), "dec_self_out": ("dec", ("self_attn", "out_proj")),
    "dec_cross_q": ("dec", ("cross_attn", "q_proj")), "dec_cross_kv": ("dec", ("cross_attn", "kv_proj")),
    "dec_cross_out": ("dec", ("cross_attn", "out_proj")),
    "dec_ffn_in": ("dec", ("ffn", "net", "0")), "dec_ffn_out": ("dec", ("ffn", "net", "2")),
}

def model_linears(backbone, which):
    """Yield (parent, attr, module) for each chosen linear in every encoder / decoder block."""
    c = backbone.core
    for w in which:
        assert w in _WHERE, f"unknown target {w}; choose from {all_target_names(backbone)}"
        side, path = _WHERE[w]
        for blk in (c.encoder_layers if side == "enc" else c.decoder_layers):
            parent = blk
            for p in path[:-1]:
                parent = getattr(parent, p)
            yield parent, path[-1], getattr(parent, path[-1])

def wrap_linears(backbone, which, make_wrapper):
    """Replace each chosen nn.Linear by make_wrapper(linear). Returns how many were wrapped."""
    todo = list(model_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A and E0 to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


# -------------------------------------------------------------------------------------
# Windows (Phase 2/4/5 make_windows + select_windows, unchanged)
# -------------------------------------------------------------------------------------
DOC_REPRS = ("all", "first", "last", "first_last", "spread")
FUSIONS = ("fid", "concat")

def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]

def select_windows(wins, doc_repr, max_windows):
    n = len(wins)
    if doc_repr == "all" or n <= max_windows:
        return wins
    if doc_repr == "first":
        return wins[:max_windows]
    if doc_repr == "last":
        return wins[-max_windows:]
    if doc_repr == "first_last":
        h = (max_windows + 1) // 2
        return wins[:h] + wins[n - (max_windows - h):] if max_windows > h else wins[:h]
    if doc_repr == "spread":
        idx = sorted({round(i * (n - 1) / max(1, max_windows - 1)) for i in range(max_windows)})
        return [wins[i] for i in idx]
    raise ValueError(doc_repr)

def doc_source(ids, window, stride, doc_repr, max_windows, fusion):
    """-> list of encoder inputs (token lists) for ONE document.
    fid: one entry per selected window. concat: the selected windows' tokens joined into one entry
    (overlapping parts of neighbouring windows are kept once)."""
    wins = select_windows(make_windows(len(ids), window, stride), doc_repr, max_windows)
    if fusion == "fid":
        return [ids[s:e] for s, e in wins]
    out, last_end = [], 0
    for s, e in wins:
        s = max(s, last_end)
        out += ids[s:e]
        last_end = e
    return [out[:MAX_POS]]

def make_target(sum_ids, dec_len):
    """Summary tokens (cut to dec_len - 1) + </s>; decoder input = <s> + target[:-1]."""
    tgt = sum_ids[:dec_len - 1] + [EOS_ID]
    return [BOS_ID] + tgt[:-1], tgt

def build_examples(docs, window, stride, doc_repr, max_windows, fusion, dec_len):
    """One example per document: {"doc", "src": [token lists], "dec_in", "target"} (dec_in/target None for test)."""
    assert window <= MAX_POS and stride <= window and doc_repr in DOC_REPRS and fusion in FUSIONS
    assert doc_repr == "all" or max_windows, "set MAX_WINDOWS for this DOC_REPR"
    assert dec_len <= MAX_POS
    if fusion == "concat":
        assert doc_repr != "all", "concat needs a window limit (DOC_REPR != 'all')"
        assert window * max_windows <= MAX_POS or doc_repr == "first", \
            f"concat: MAX_WINDOWS x WINDOW = {window * max_windows} > {MAX_POS} (anything past {MAX_POS} is cut)"
    ex, n_cut = [], 0
    for k, d in tqdm(docs.items(), desc="examples", leave=False):
        e = {"doc": k, "src": doc_source(d["ids"], window, stride, doc_repr, max_windows, fusion),
             "dec_in": None, "target": None}
        if d.get("sum_ids") is not None:
            e["dec_in"], e["target"] = make_target(d["sum_ids"], dec_len)
            n_cut += len(d["sum_ids"]) > dec_len - 1
        ex.append(e)
    nw = np.array([len(e["src"]) for e in ex])
    nt = np.array([sum(map(len, e["src"])) for e in ex])
    nd = np.array([len(d["ids"]) for d in docs.values()])
    line = (f"{len(ex)} docs | encoder pieces per doc mean {nw.mean():.2f} max {nw.max()} | "
            f"encoder tokens per doc median {np.median(nt):.0f} (whole doc {np.median(nd):.0f}) | "
            f"doc tokens seen {nt.sum() / nd.sum():.1%}")
    if ex[0]["target"] is not None:
        line += f" | summaries cut at DEC_LEN: {n_cut} ({n_cut / len(ex):.1%})"
    print(line)
    return ex

def make_batches(examples, batch_size, shuffle):
    """Documents of similar encoder size go together (less padding); batch order shuffled for training."""
    order = sorted(range(len(examples)), key=lambda i: (sum(map(len, examples[i]["src"])),
                                                         len(examples[i]["target"] or [])))
    batches = [[examples[i] for i in order[s:s + batch_size]] for s in range(0, len(order), batch_size)]
    if shuffle:
        random.shuffle(batches)
    return batches

def _pad(rows, value):
    T = max(len(r) for r in rows)
    return torch.tensor([r + [value] * (T - len(r)) for r in rows], dtype=torch.long, device=device)

def to_tensors(batch):
    """Flattens every document's encoder pieces into one padded (N, T) batch; doc_of[n] = doc index."""
    pieces = [p for e in batch for p in e["src"]]
    b = {"enc": _pad(pieces, PAD_ID),
         "piece_lens": [len(p) for p in pieces],
         "n_pieces": [len(e["src"]) for e in batch]}
    b["enc_mask"] = None                                  # no mask needed when every piece has the same length
    if min(b["piece_lens"]) < b["enc"].size(1):
        L = torch.tensor(b["piece_lens"], device=device)
        b["enc_mask"] = torch.arange(b["enc"].size(1), device=device)[None, :] < L[:, None]   # True = real token
    if batch[0]["target"] is not None:
        b["dec_in"] = _pad([e["dec_in"] for e in batch], PAD_ID)
        b["target"] = _pad([e["target"] for e in batch], IGNORE)
    return b


class Seq2SeqModel(nn.Module):
    """c-6 encoder-decoder for summarisation. memory(b) = all encoder pieces of each document,
    concatenated ("fid") or single ("concat"), padded across the batch with a mask."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def memory(self, b):
        h = encoder_forward(self.backbone, b["enc"], b["enc_mask"])          # (N, T, 560)
        if all(n == 1 for n in b["n_pieces"]) and b["enc_mask"] is None:
            return h, None
        rows, i = [], 0
        for n in b["n_pieces"]:
            rows.append(torch.cat([h[i + j, :b["piece_lens"][i + j]] for j in range(n)], dim=0))
            i += n
        M = max(r.size(0) for r in rows)
        mem = h.new_zeros(len(rows), M, h.size(-1))
        mask = torch.zeros(len(rows), M, dtype=torch.bool, device=h.device)
        for r, x in enumerate(rows):
            mem[r, :x.size(0)] = x
            mask[r, :x.size(0)] = True
        return mem, (None if bool(mask.all()) else mask)

    def forward(self, b):
        mem, mask = self.memory(b)
        return decoder_forward(self.backbone, b["dec_in"], mem, mask)       # (B, Td, V)


def seq2seq_loss(logits, b, label_smoothing=0.0):
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), b["target"].reshape(-1),
                           ignore_index=IGNORE, label_smoothing=label_smoothing or 0.0)


def _split(batch, micro):
    if not micro or micro >= len(batch):
        return [batch]
    return [batch[i:i + micro] for i in range(0, len(batch), micro)]


@torch.no_grad()
def val_loss(model, examples, batch_size):
    """Mean token cross-entropy (no label smoothing) on the held-out summaries -> (loss, perplexity)."""
    model.eval()
    tot, n = 0.0, 0
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="val loss", leave=False):
        b = to_tensors(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(b)
        tot += F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), b["target"].reshape(-1),
                               ignore_index=IGNORE, reduction="sum").item()
        n += (b["target"] != IGNORE).sum().item()
    l = tot / max(1, n)
    return l, math.exp(min(l, 50))


# -------------------------------------------------------------------------------------
# Generation: encode once, cache cross-attention K/V and self-attention K/V per layer,
# one decoder step per new token. Decoding controls (all set in Cell E2):
#   max_new_tokens, min_new_tokens (</s> is blocked before this), do_sample (False = greedy),
#   temperature / top_k / top_p (sampling only), repetition_penalty (1.0 = off),
#   no_repeat_ngram (0 = off), max_words (cut the decoded text to this many words, None = off)
# -------------------------------------------------------------------------------------
def _decode_step(backbone, tok_t, pos, caches, xkv, mem_mask):
    c = backbone.core
    x = c.tok_emb(tok_t) + c.pos_emb(torch.full_like(tok_t, pos))
    for blk, cache, kv in zip(c.decoder_layers, caches, xkv):
        x = x + dec_self_attn(blk.self_attn, blk.ln1(x), cache)
        x = x + cross_attn(blk.cross_attn, blk.ln2(x), kv, mem_mask)
        x = x + blk.ffn(blk.ln3(x))
    return c.lm_head(c.decoder_ln(x))[:, -1, :].float()                     # (B, V)

def _no_repeat(logits, gen, n):
    t = gen.size(1)
    if n < 2 or t < n - 1:
        return logits
    for r in range(gen.size(0)):
        seq = gen[r].tolist()
        tail = tuple(seq[t - (n - 1):])
        banned = {seq[i + n - 1] for i in range(t - n + 1) if tuple(seq[i:i + n - 1]) == tail}
        if banned:
            logits[r, list(banned)] = float("-inf")
    return logits

def _pick(logits, do_sample, temperature, top_k, top_p):
    if not do_sample:
        return logits.argmax(-1, keepdim=True)
    if top_k:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    probs = F.softmax(logits / max(temperature or 1.0, 1e-5), dim=-1)
    if top_p is not None and top_p < 1.0:
        sp, si = torch.sort(probs, descending=True, dim=-1)
        sp = sp.masked_fill(sp.cumsum(-1) - sp > top_p, 0.0)
        sp = sp / sp.sum(-1, keepdim=True)
        return si.gather(-1, torch.multinomial(sp, 1))
    return torch.multinomial(probs, 1)

@torch.no_grad()
def generate(model, b, max_new_tokens, min_new_tokens=0, do_sample=False, temperature=1.0, top_k=None,
             top_p=None, repetition_penalty=1.0, no_repeat_ngram=0):
    """-> list of generated token lists (without <s>, cut at </s>)."""
    model.eval()
    bb = model.backbone
    assert max_new_tokens <= MAX_POS, f"max_new_tokens > {MAX_POS} (decoder position table)"
    with torch.autocast("cuda", dtype=torch.float16):
        mem, mem_mask = model.memory(b)
        xkv = [cross_kv(blk.cross_attn, mem) for blk in bb.core.decoder_layers]
    B = mem.size(0)
    caches = [dict() for _ in bb.core.decoder_layers]
    tok_t = torch.full((B, 1), BOS_ID, dtype=torch.long, device=device)
    gen = torch.zeros((B, 0), dtype=torch.long, device=device)
    done = torch.zeros(B, dtype=torch.bool, device=device)
    for t in range(max_new_tokens):
        with torch.autocast("cuda", dtype=torch.float16):
            logits = _decode_step(bb, tok_t, t, caches, xkv, mem_mask)
        logits[:, [PAD_ID, UNK_ID, BOS_ID]] = float("-inf")
        if t < (min_new_tokens or 0):
            logits[:, EOS_ID] = float("-inf")
        if repetition_penalty and repetition_penalty != 1.0 and gen.size(1):
            seen = torch.zeros_like(logits, dtype=torch.bool).scatter_(1, gen, True)
            seen[:, EOS_ID] = False
            pen = torch.where(logits > 0, logits / repetition_penalty, logits * repetition_penalty)
            logits = torch.where(seen, pen, logits)
        logits = _no_repeat(logits, gen, no_repeat_ngram or 0)
        nxt = _pick(logits, do_sample, temperature, top_k, top_p)
        nxt = torch.where(done[:, None], torch.full_like(nxt, EOS_ID), nxt)
        gen = torch.cat([gen, nxt], dim=1)
        done |= nxt[:, 0] == EOS_ID
        tok_t = nxt
        if bool(done.all()):
            break
    out = []
    for row in gen.tolist():
        out.append(row[:row.index(EOS_ID)] if EOS_ID in row else row)
    return out

def predict(model, examples, gen_batch_size, gen_kw, max_words=None, desc="generate"):
    """-> {doc_id: summary text}."""
    pred = {}
    t0 = time.time()
    for batch in tqdm(make_batches(examples, gen_batch_size, shuffle=False), desc=desc):
        for e, ids in zip(batch, generate(model, to_tensors(batch), **gen_kw)):
            txt = ids_to_text(ids)
            pred[e["doc"]] = " ".join(txt.split()[:max_words]) if max_words else txt
    print(f"{desc}: {len(pred)} docs in {(time.time() - t0) / 60:.1f} min")
    return pred


# -------------------------------------------------------------------------------------
# Training: AdamW on the trainable parameters only (fp16 autocast + GradScaler), optional
# micro-batching (same update, less memory), grad clipping, per-epoch time cap.
# Best epoch chosen by select_by on val:
#   "loss"   = val token cross-entropy (cheap, no generation)
#   "rougeL" / "rouge1" / "bertscore" = generate val_gen_n held-out docs each epoch and score them
# Saves best_trainable.pt (trainable tensors of the best epoch) and last_checkpoint.pt
# (everything needed to resume: trainable, best, optimizer, scaler, scheduler, epoch, log).
# -------------------------------------------------------------------------------------
def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_model(model, train_ex, val_ex, val_docs, lr, epochs, batch_size, weight_decay, grad_clip=None,
                label_smoothing=0.0, lr_lambda=None, save_path="best_trainable.pt", last_path=None,
                resume_from=None, micro_batch=None, max_epoch_minutes=None, select_by="loss",
                val_gen_n=None, gen_batch_size=8, gen_kw=None, max_words=None):
    assert select_by in ("loss", "rougeL", "rouge1", "bertscore")
    assert select_by != "bertscore" or _bs_mod is not None, "bert-score is not installed: pick another SELECT_BY"
    params = [p for p in model.parameters() if p.requires_grad]
    assert params, "no trainable parameters"
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scaler = torch.cuda.amp.GradScaler()
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    best, log, start = -float("inf"), [], 0
    if resume_from:
        ck = torch.load(resume_from, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if sched and ck.get("sched"):
            sched.load_state_dict(ck["sched"])
        best, log, start = ck["best"], ck["log"], ck["epoch"]
        torch.save(ck["best_trainable"], save_path)
        print(f"resumed after epoch {start} (best {select_by} {abs(best):.4f})")
        del ck
    gen_ids = set(list(val_docs)[:val_gen_n]) if (select_by != "loss" and val_gen_n) else set(val_docs)
    gen_ex = [e for e in val_ex if e["doc"] in gen_ids]
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    for ep in range(start, epochs):
        model.train()
        t0, tot, ntok, seen = time.time(), 0.0, 0, 0
        batches = make_batches(train_ex, batch_size, shuffle=True)
        pbar = tqdm(batches, desc=f"epoch {ep + 1}/{epochs}")
        for batch in pbar:
            opt.zero_grad(set_to_none=True)
            parts = _split(batch, micro_batch)
            n_all = sum(len(e["target"]) for e in batch)
            for part in parts:
                b = to_tensors(part)
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(b)
                n_part = (b["target"] != IGNORE).sum().item()
                loss = seq2seq_loss(logits, b, label_smoothing)
                scaler.scale(loss * n_part / n_all).backward()        # token-weighted, same as one big batch
                tot += loss.item() * n_part; ntok += n_part
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched:
                sched.step()
            seen += 1
            pbar.set_postfix(loss=f"{tot / max(1, ntok):.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")
            if max_epoch_minutes and time.time() - t0 > 60 * max_epoch_minutes:
                print(f"  time cap: stopped epoch after {seen}/{len(batches)} batches")
                break
        vl, ppl = val_loss(model, val_ex, batch_size)
        row = {"epoch": ep + 1, "train_loss": tot / max(1, ntok), "val_loss": vl, "val_ppl": ppl,
               "batches": seen, "minutes": (time.time() - t0) / 60}
        if select_by == "loss":
            score = -vl
        else:
            p = predict(model, gen_ex, gen_batch_size, gen_kw, max_words, desc="val generate")
            r = evaluate_summaries(p, {k: val_docs[k] for k in p}, show=True,
                                   with_bertscore=select_by == "bertscore")
            row.update({k: r[k] for k in ("rouge1_f1", "rougeL_f1", "bertscore_f1", "pred_words_median")})
            score = {"rougeL": r["rougeL_f1"], "rouge1": r["rouge1_f1"], "bertscore": r["bertscore_f1"]}[select_by]
        log.append(row)
        improved = score > best
        if improved:
            best = score
            torch.save(trainable_state(model), save_path)
        print(f"epoch {ep + 1}: train loss {row['train_loss']:.4f} | val loss {vl:.4f} (ppl {ppl:.1f}) | "
              f"{select_by} {abs(score):.4f}{' *best*' if improved else ''} | {row['minutes']:.1f} min")
        if last_path:
            torch.save({"trainable": trainable_state(model), "best_trainable": torch.load(save_path),
                        "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict() if sched else None, "epoch": ep + 1, "best": best,
                        "log": log}, last_path)
    return log, best


# -------------------------------------------------------------------------------------
# Long-document analysis helpers (for the report)
# -------------------------------------------------------------------------------------
_WORD = re.compile(r"[a-z][a-z0-9]{3,}")

def coverage_report(docs, examples):
    """How much of each reference summary's vocabulary (words of 4+ letters) the model can SEE:
    in the selected encoder input vs in the whole document. The gap is information the
    long-document strategy throws away before the model even starts."""
    by = {e["doc"]: e for e in examples}
    cov_in, cov_doc = [], []
    for k, d in docs.items():
        if not d.get("summary") or k not in by:
            continue
        sw = set(_WORD.findall(d["summary"].lower()))
        if not sw:
            continue
        seen = set(_WORD.findall(" ".join(tok.decode(p) for p in by[k]["src"]).lower()))
        full = set(_WORD.findall(d["text"].lower()))
        cov_in.append(len(sw & seen) / len(sw)); cov_doc.append(len(sw & full) / len(sw))
    print(f"summary words found in the encoder input {np.mean(cov_in):.1%} vs in the whole document "
          f"{np.mean(cov_doc):.1%} (n={len(cov_in)})")
    return float(np.mean(cov_in)), float(np.mean(cov_doc))

def show_examples(pred, docs, n=2, words=120):
    for k in list(pred)[:n]:
        print(f"--- {k}\nPRED: {' '.join(pred[k].split()[:words])}")
        if docs[k].get("summary"):
            print(f"REF : {' '.join(docs[k]['summary'].split()[:words])}")


# -------------------------------------------------------------------------------------
# Self-checks: the masked / cached paths equal the original Phase 1 model
# -------------------------------------------------------------------------------------
def _self_checks():
    lm.eval()
    d = next(iter(train.values()))
    src = torch.tensor([d["ids"][:64]], device=device)
    dec = torch.tensor([[BOS_ID] + d["sum_ids"][:31]], device=device)
    with torch.no_grad():
        ref_enc = lm.encode(src)
        ref_logits = lm.decode(dec, ref_enc)
        e1 = (encoder_forward(lm, src) - ref_enc).abs().max().item()
        e2 = (decoder_forward(lm, dec, ref_enc) - ref_logits).abs().max().item()
        # padding invariance: the same 40 tokens padded to 64 with a mask
        short = src[:, :40]
        padded = torch.cat([short, torch.full((1, 24), PAD_ID, device=device)], dim=1)
        mask = torch.arange(64, device=device)[None, :] < 40
        e3 = (encoder_forward(lm, padded, mask)[:, :40] - lm.encode(short)).abs().max().item()
        mem_p = torch.cat([lm.encode(short), torch.randn(1, 24, FEAT_DIM, device=device)], dim=1)
        e4 = (decoder_forward(lm, dec, mem_p, mask) - lm.decode(dec, lm.encode(short))).abs().max().item()
        # KV-cached step decoding vs the full teacher-forced decoder
        caches = [dict() for _ in lm.core.decoder_layers]
        xkv = [cross_kv(blk.cross_attn, ref_enc) for blk in lm.core.decoder_layers]
        steps = torch.stack([_decode_step(lm, dec[:, t:t + 1], t, caches, xkv, None) for t in range(dec.size(1))], 1)
        e5 = (steps - ref_logits.float()).abs().max().item()
    print(f"self-checks max |diff|: encoder {e1:.1e} | decoder {e2:.1e} | padded encoder {e3:.1e} | "
          f"masked cross-attn {e4:.1e} | cached decoding {e5:.1e}  (all should be ~0)")
    assert max(e1, e2, e3, e4, e5) < 1e-3, "masked / cached forward differs from the Phase 1 model"

_self_checks()

self-checks max |diff|: encoder 9.5e-07 | decoder 3.8e-06 | padded encoder 9.5e-07 | masked cross-attn 3.8e-06 | cached decoding 4.8e-06  (all should be ~0)


## Cell E2 — LORMA  (fill section [H])

In [9]:
# =====================================================================================
# CELL E2 — LORMA (multiplicative low-rank adaptation, NOT LoRA) on the c-6 encoder-decoder
#           for Phase 6 abstractive summarisation
#
# LORMA:  W' = (I + B A) W      -> y = W x + B A (W x) + b      (multiplicative update of W)
# LoRA :  W' =  W + B A         -> y = W x + B A x + b          (additive; NOT what this cell does)
# The layer below is your own Phase 3/5 LORMALinear, unchanged.
#
# Run after Cells 0, A, B, C, D and E0. To run again from scratch, re-run Cell A and E0 first
# (wrapping replaces the linears in place, and wrap_linears refuses to wrap twice).
#
# >>> FILL IN: every value marked  # <<< FILL  in section [H]. Nothing else needs changing. <<<
# Sections marked [ANCILLARY] are plumbing.
# =====================================================================================

# -------------------------------------------------------------------------------------
# [H] Hyperparameters and design choices — YOURS (all left unset on purpose)
# -------------------------------------------------------------------------------------
# long document -> encoder
WINDOW        = 1024   # <<< FILL  tokens per encoder window (<= 1536)
STRIDE        = 512   # <<< FILL  step between window starts (<= WINDOW)
DOC_REPR      = "spread"   # <<< FILL  which windows: "all", "first", "last", "first_last" or "spread"
MAX_WINDOWS   = 4   # <<< FILL  windows kept per document (needed unless DOC_REPR == "all")
FUSION        = "fid"   # <<< FILL  "fid" (encode windows separately, decoder attends to all) or "concat" (one input, MAX_WINDOWS x WINDOW <= 1536)
DEC_LEN       = 256   # <<< FILL  summary tokens trained on, incl. </s> (<= 1536; longer summaries are cut)

# LORMA
LORMA_RANK    = 8   # <<< FILL
LORMA_TARGETS = ("enc_qkv", "enc_out", "enc_ffn_in", "enc_ffn_out", "dec_self_q", "dec_self_k", "dec_self_v", "dec_self_out", "dec_cross_q", "dec_cross_kv", "dec_cross_out", "dec_ffn_in", "dec_ffn_out")   # <<< FILL  tuple of names from the list printed below, e.g. encoder and/or decoder linears

# training
LR            = 1e-4   # <<< FILL
EPOCHS        = 10   # <<< FILL  (total; with RESUME_FROM, the epochs already done count)
BATCH_SIZE    = 4   # <<< FILL  documents per optimizer step
WEIGHT_DECAY  = 0.01   # <<< FILL
GRAD_CLIP     = 1.0   # <<< FILL  max grad norm (0 = off)
LABEL_SMOOTHING = 0.05 # <<< FILL  (0.0 = off)
LR_LAMBDA     = None   # optional: step -> LR multiplier (None = constant LR)
MAX_EPOCH_MINUTES = None   # optional time cap per epoch (None = no cap)
SELECT_BY     = "loss"   # <<< FILL  best epoch by "loss" (val cross-entropy), "rougeL", "rouge1" or "bertscore"
VAL_GEN_N     = None   # <<< FILL  if SELECT_BY != "loss": held-out docs generated each epoch (None = all val)

# decoding (changing only these does not count as a new experiment)
GEN_BATCH_SIZE     = 4   # <<< FILL
MAX_NEW_TOKENS     = 220   # <<< FILL  (<= 1536)
MIN_NEW_TOKENS     = 20   # <<< FILL  </s> is blocked before this many tokens (0 = off)
DO_SAMPLE          = False   # <<< FILL  False = greedy, True = sampling
TEMPERATURE        = 1.0   # <<< FILL  sampling only (ignored when DO_SAMPLE is False)
TOP_K              = None   # <<< FILL  sampling only, None / 0 = off
TOP_P              = 1.0   # <<< FILL  sampling only, 1.0 = off
REPETITION_PENALTY = 1.1   # <<< FILL  1.0 = off
NO_REPEAT_NGRAM    = 3   # <<< FILL  0 = off
MAX_SUMMARY_WORDS  = 150   # optional: cut each generated summary to this many words (None = off)

SEED          = 42   # <<< FILL

# [ANCILLARY] GPU memory savers: same maths and same updates, only slower. Use them on CUDA OutOfMemoryError.
GRAD_CHECKPOINT = False    # True = recompute each encoder/decoder block in backward instead of storing it
MICRO_BATCH     = 1     # e.g. 2 = run each batch 2 documents at a time, one optimizer step per batch (None = off)
# [ANCILLARY] continue an earlier run: path to its last_checkpoint.pt (attach that notebook's output as input;
# find it with  !find /kaggle/input -name last_checkpoint.pt ). None = fresh start.
RESUME_FROM     = None
# [ANCILLARY] also generate summaries for the TRAIN documents with the final model (teacher
# summaries for a sequence-level KD notebook later; slow: one generation per train doc).
DUMP_TRAIN_SUMMARIES = False

_required = ["WINDOW", "STRIDE", "DOC_REPR", "FUSION", "DEC_LEN", "LORMA_RANK", "LORMA_TARGETS", "LR", "EPOCHS",
             "BATCH_SIZE", "WEIGHT_DECAY", "GRAD_CLIP", "LABEL_SMOOTHING", "SELECT_BY", "GEN_BATCH_SIZE",
             "MAX_NEW_TOKENS", "MIN_NEW_TOKENS", "DO_SAMPLE", "REPETITION_PENALTY", "NO_REPEAT_NGRAM", "SEED"]
if DO_SAMPLE:
    _required += ["TEMPERATURE", "TOP_P"]
_unset = [n for n in _required if globals()[n] is None]
assert not _unset, f"fill in section [H] first: {_unset}"
assert STRIDE <= WINDOW <= cfg.max_seq_len and DEC_LEN <= cfg.max_seq_len and MAX_NEW_TOKENS <= cfg.max_seq_len
assert DOC_REPR == "all" or MAX_WINDOWS, "set MAX_WINDOWS"
assert isinstance(LORMA_TARGETS, (tuple, list)) and LORMA_TARGETS, "LORMA_TARGETS must be a non-empty tuple"


# -------------------------------------------------------------------------------------
# [ANCILLARY] run setup
# -------------------------------------------------------------------------------------
BUILD = dict(window=WINDOW, stride=STRIDE, doc_repr=DOC_REPR, max_windows=MAX_WINDOWS, fusion=FUSION, dec_len=DEC_LEN)
GEN_KW = dict(max_new_tokens=MAX_NEW_TOKENS, min_new_tokens=MIN_NEW_TOKENS, do_sample=DO_SAMPLE,
              temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
              repetition_penalty=REPETITION_PENALTY, no_repeat_ngram=NO_REPEAT_NGRAM)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c6_lorma_summ_{FUSION}_{DOC_REPR}"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"; os.makedirs(OUT_DIR, exist_ok=True)
SAVE = f"{OUT_DIR}/best_trainable.pt"
LAST = f"{OUT_DIR}/last_checkpoint.pt"

# Which linears can get LORMA (sizes from the loaded model, block 0; all 8 blocks per side are the same)
print("LORMA target names (encoder: enc_*, decoder: dec_*):")
for _name in all_target_names(lm):
    _parent, _attr, _lin = next(model_linears(lm, (_name,)))
    print(f"  {_name:14s} {_lin.in_features:5d} -> {_lin.out_features:5d}  bias={_lin.bias is not None}")


# -------------------------------------------------------------------------------------
# [L] LORMA layer — yours (Phase 3/5): W' = (I + BA) W   (multiplicative, not LoRA's W + BA)
# -------------------------------------------------------------------------------------
class LORMALinear(nn.Module):
    """Wraps ONE pretrained nn.Linear of the c-6 encoder or decoder.
    forward(x): (..., in_features) -> (..., out_features), same shape as base(x).
    Pretrained weights frozen; only lorma_A / lorma_B train. B = 0 at init, so the wrapped
    layer starts exactly equal to the pretrained one (identity check below)."""
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        # Freeze pretrained weights
        for p in self.base.parameters():
            p.requires_grad = False

        # A: (rank, out_features), since A operates on Wx
        # B: (out_features, rank)
        self.lorma_A = nn.Parameter(
            torch.empty(self.rank, self.out_features)
        )
        self.lorma_B = nn.Parameter(
            torch.empty(self.out_features, self.rank)
        )

        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        # Compute pretrained linear output without bias: Wx
        wx = F.linear(x, self.base.weight, bias=None)

        # Low-rank transformation of Wx: BA(Wx)
        lorma_out = F.linear(
            F.linear(wx, self.lorma_A),
            self.lorma_B
        )

        # Return Wx + BA(Wx) + b   ==  (I + BA) W x + b
        if self.base.bias is not None:
            return wx + lorma_out + self.base.bias

        return wx + lorma_out


# [ANCILLARY] quick checks on a dummy linear: shape, and that it really is multiplicative LORMA
_t = LORMALinear(nn.Linear(560, 1680, bias=False), rank=LORMA_RANK)
_x = torch.randn(2, 7, 560)
assert _t(_x).shape == (2, 7, 1680)
with torch.no_grad():
    nn.init.normal_(_t.lorma_B, std=0.02)
    _W = _t.base.weight
    _merged = (torch.eye(1680) + _t.lorma_B @ _t.lorma_A) @ _W          # (I + BA) W
    assert torch.allclose(_t(_x), _x @ _merged.T, atol=1e-4), "layer is not (I + BA) W"
print("LORMALinear = (I + BA)W checked | trainable params on a 560->1680 layer:",
      sum(p.numel() for p in _t.parameters() if p.requires_grad))
del _t, _x, _W, _merged


# -------------------------------------------------------------------------------------
# [ANCILLARY] freeze c-6, wrap the chosen linears with LORMA, check identity at init
# -------------------------------------------------------------------------------------
GRAD_CHECKPOINT = GRAD_CHECKPOINT       # read by encoder_forward / decoder_forward in E0
lm.eval()
_d = next(iter(train.values()))
probe_src = torch.tensor([_d["ids"][:64]], device=device)
probe_dec = torch.tensor([[BOS_ID] + _d["sum_ids"][:31]], device=device)
with torch.no_grad():
    ref = lm.decode(probe_dec, lm.encode(probe_src)).float()

for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(lm, LORMA_TARGETS, lambda lin: LORMALinear(lin, LORMA_RANK).to(device))
print(f"wrapped {n_wrapped} linears with LORMA ({len(LORMA_TARGETS)} target names x {cfg.n_layers} blocks)")
assert all(isinstance(m, LORMALinear) for _, _, m in model_linears(lm, LORMA_TARGETS))

with torch.no_grad():
    diff = (decoder_forward(lm, probe_dec, encoder_forward(lm, probe_src)).float() - ref).abs().max().item()
print(f"LORMA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
assert diff < 1e-3, "wrapped model differs from the original at init"

model = Seq2SeqModel(lm).to(device)
count_params(model)
_lorma_n = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and "lorma_" in n)
_other = [n for n, p in model.named_parameters() if p.requires_grad and "lorma_" not in n]
assert not _other, f"only LORMA parameters should train, found {_other[:3]}"
print(f"all {_lorma_n/1e6:.3f}M trainable parameters are LORMA A/B matrices")


# -------------------------------------------------------------------------------------
# [ANCILLARY] data, long-document coverage, training, val scoring, test submission
# -------------------------------------------------------------------------------------
train_ex = build_examples(train, **BUILD)
val_ex = build_examples(val, **BUILD)
test_ex = build_examples(test, **BUILD)
print("\n-- long-document strategy: what the encoder sees (val)")
cov = coverage_report(val, val_ex)

t_train = time.time()
log, best = train_model(
    model, train_ex, val_ex, val,
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    label_smoothing=LABEL_SMOOTHING, lr_lambda=LR_LAMBDA, save_path=SAVE, last_path=LAST,
    resume_from=RESUME_FROM, micro_batch=MICRO_BATCH, max_epoch_minutes=MAX_EPOCH_MINUTES,
    select_by=SELECT_BY, val_gen_n=VAL_GEN_N, gen_batch_size=GEN_BATCH_SIZE, gen_kw=GEN_KW,
    max_words=MAX_SUMMARY_WORDS)
train_min = (time.time() - t_train) / 60
train_mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0

model.load_state_dict(torch.load(SAVE), strict=False)
t0 = time.time()
val_pred = predict(model, val_ex, GEN_BATCH_SIZE, GEN_KW, MAX_SUMMARY_WORDS, desc="val generate")
val_gen_s = time.time() - t0
print("\n-- held-out val (best epoch)")
res = evaluate_summaries(val_pred, val)
length_buckets(res)
print(f"3-gram repetition rate: pred {repetition_rate(val_pred.values()):.3f} | "
      f"reference {repetition_rate(d['summary'] for d in val.values()):.3f}")
print(f"lead-{LEAD_N} baseline: BERTScore {baseline_res['bertscore_f1']} | R-1 {baseline_res['rouge1_f1']:.4f} | "
      f"R-L {baseline_res['rougeL_f1']:.4f}")
show_examples(val_pred, val)
print(f"training {train_min:.1f} min | val generation {val_gen_s:.0f} s for {len(val_pred)} docs | "
      f"peak training memory {train_mem:.2f} GB")
json.dump({"run": RUN_NAME, "method": "LORMA (I+BA)W", "build": BUILD, "gen": GEN_KW,
           "lorma_rank": LORMA_RANK, "lorma_targets": list(LORMA_TARGETS), "log": log,
           "coverage_input_vs_doc": cov, "train_minutes": train_min, "peak_mem_gb": train_mem,
           "val": {k: res[k] for k in ("bertscore_f1", "rouge1_f1", "rougeL_f1", "pred_words_median",
                                       "ref_words_median", "len_ratio_median")},
           "val_pred": val_pred}, open(f"{OUT_DIR}/val_log.json", "w"), ensure_ascii=False)

# Test summaries with the best weights -> /kaggle/working/<RUN_NAME>/submission.zip
test_pred = predict(model, test_ex, GEN_BATCH_SIZE, GEN_KW, MAX_SUMMARY_WORDS, desc="test generate")
write_prediction(test_pred, out_dir=OUT_DIR)

# [ANCILLARY] everything a KD notebook needs to rebuild this model as the teacher
if DUMP_TRAIN_SUMMARIES:
    train_pred = predict(model, train_ex, GEN_BATCH_SIZE, GEN_KW, MAX_SUMMARY_WORDS, desc="train generate")
    json.dump(train_pred, open(f"{OUT_DIR}/teacher_train_summaries.json", "w"), ensure_ascii=False)
json.dump({"run": RUN_NAME, "method": "LORMA (I+BA)W", "build": BUILD, "gen": GEN_KW,
           "lorma_rank": LORMA_RANK, "lorma_targets": list(LORMA_TARGETS), "weights": "best_trainable.pt",
           "val_ids": sorted(VAL_IDS)}, open(f"{OUT_DIR}/teacher_config.json", "w"), indent=1)
print("files for later runs / KD:", sorted(os.listdir(OUT_DIR)))

LORMA target names (encoder: enc_*, decoder: dec_*):
  enc_qkv          560 ->  1680  bias=False
  enc_out          560 ->   560  bias=False
  enc_ffn_in       560 ->  1000  bias=True
  enc_ffn_out     1000 ->   560  bias=True
  dec_self_q       560 ->   560  bias=False
  dec_self_k       560 ->   560  bias=False
  dec_self_v       560 ->   560  bias=False
  dec_self_out     560 ->   560  bias=False
  dec_cross_q      560 ->   560  bias=False
  dec_cross_kv     560 ->  1120  bias=False
  dec_cross_out    560 ->   560  bias=False
  dec_ffn_in       560 ->  1000  bias=True
  dec_ffn_out     1000 ->   560  bias=True
LORMALinear = (I + BA)W checked | trainable params on a 560->1680 layer: 26880
wrapped 104 linears with LORMA (13 target names x 8 blocks)
LORMA identity-at-init max |diff| = 3.81e-06 (should be ~0)
trainable 1.260M / total 56.39M (2.23%)
all 1.260M trainable parameters are LORMA A/B matrices


examples:   0%|          | 0/6325 [00:00<?, ?it/s]

6325 docs | encoder pieces per doc mean 3.72 max 4 | encoder tokens per doc median 4096 (whole doc 3903) | doc tokens seen 69.9% | summaries cut at DEC_LEN: 6111 (96.6%)


examples:   0%|          | 0/703 [00:00<?, ?it/s]

703 docs | encoder pieces per doc mean 3.74 max 4 | encoder tokens per doc median 4096 (whole doc 3976) | doc tokens seen 66.8% | summaries cut at DEC_LEN: 681 (96.9%)


examples:   0%|          | 0/100 [00:00<?, ?it/s]

... (38 outputs left out)

repetition:   0%|          | 0/703 [00:00<?, ?it/s]

3-gram repetition rate: pred 0.032 | reference 0.084
lead-637 baseline: BERTScore 0.8311378359794617 | R-1 0.4939 | R-L 0.2473
--- 5156
PRED: The appellant was a suit for the appellant and the appellant was convicted under section 302 of the Indian Penal Code of the High Court held that the appellant had been in the appellant to be a person. The High Court dismissed the appeal by the High Court. The High Court, HELD: The High Court. In this Court is not a case of the case of a case in the case in view of the court, it is not only if the case is not be considered. The question of the appeal is not in the High Court, : (i) the case must be in the evidence of the conclusion that the case should be made by the case and
REF : HELD: 1. The fact that the offences, if any, are alleged to have been committed more than six years ago and the appellant was merely a trader at the lowest rung of the hierarchy in the Foreign Exchange Division of the Bank and not a highly placed officer and the trial 

test generate:   0%|          | 0/25 [00:00<?, ?it/s]

test generate: 100 docs in 2.3 min
wrote /kaggle/working/c6_lorma_summ_fid_spread/prediction.json and /kaggle/working/c6_lorma_summ_fid_spread/submission.zip: 100 summaries | words min 138 median 150 max 150 | empty (written as '.') 0
files for later runs / KD: ['best_trainable.pt', 'last_checkpoint.pt', 'prediction.json', 'submission.zip', 'teacher_config.json', 'val_log.json']
